In [1]:
# Import required packages
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Directories
PROJECT_DIR = r"C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"

CODE_DIR = os.path.join(PROJECT_DIR, "Code")
DATA_DIR = os.path.join(PROJECT_DIR, "Data")
FIGURES_DIR = os.path.join(PROJECT_DIR, "Figures")
TABLES_DIR = os.path.join(PROJECT_DIR, "Tables")

# File names
INPUT_DATA = os.path.join(DATA_DIR, "merged_master.pkl")

# Parallel processing
N_JOBS = 16  # Number of parallel jobs for OOS predictions

# Elastic Net parameters
# Both alpha and l1_ratio are determined via cross-validation
# l1_ratio = 1 is pure LASSO, l1_ratio = 0 is pure Ridge
L1_RATIOS = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0]
MAX_ITER = 10000  # Increased iterations for convergence
CV_FOLDS = 5  # Number of cross-validation folds

In [3]:
data = pd.read_pickle(INPUT_DATA)

In [4]:
# Prepare features and target
# Target variable
TARGET = 'f_cumret1'

# Select features
FEATURES = ['net_sentiment', 'log_volume']

# Remove missing values
model_data = data[[TARGET] + FEATURES].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Features: {FEATURES}")

Sample size: 16,743,676
Target: f_cumret1
Features: ['net_sentiment', 'log_volume']


In [5]:
# Add date column and sort
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])
model_data = model_data.sort_values('date')
model_data['year_month'] = model_data['date'].dt.to_period('M')

# In-Sample Elastic Net regression with optimal l1_ratio

In [6]:
# In-sample training window (adjustable)
TRAIN_START_DATE = '2011-01-01'
TRAIN_END_DATE = '2011-12-31'

# Filter data to training window
train_start = pd.to_datetime(TRAIN_START_DATE)
train_end = pd.to_datetime(TRAIN_END_DATE)
train_mask = (model_data['date'] >= train_start) & (model_data['date'] <= train_end)

X_train = model_data.loc[train_mask, FEATURES]
y_train = model_data.loc[train_mask, TARGET]

print(f"Training window: {TRAIN_START_DATE} to {TRAIN_END_DATE}")
print(f"Training samples: {len(X_train):,}")

# Normalize features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Fit Elastic Net with cross-validation for optimal alpha AND l1_ratio
import time
start_time = time.time()

enet_model = ElasticNetCV(l1_ratio=L1_RATIOS, cv=CV_FOLDS, random_state=42, n_jobs=-1, max_iter=MAX_ITER)
enet_model.fit(X_train_scaled, y_train)

elapsed_time = time.time() - start_time

# Make predictions
y_pred = enet_model.predict(X_train_scaled)

# Evaluate performance
r2 = r2_score(y_train, y_pred)
mse = mean_squared_error(y_train, y_pred)
rmse = np.sqrt(mse)

print(f"\nIn-Sample Elastic Net Regression Results")
print("=" * 50)
print(f"Optimal l1_ratio: {enet_model.l1_ratio_:.6f}")
print(f"Optimal alpha (regularization): {enet_model.alpha_:.6f}")
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print("\nCoefficients (on standardized features):")
for feature, coef in zip(FEATURES, enet_model.coef_):
    print(f"  {feature}: {coef:.6f}")
print(f"  Intercept: {enet_model.intercept_:.6f}")
print(f"\nTraining time: {elapsed_time:.2f} seconds")

Training window: 2011-01-01 to 2011-12-31
Training samples: 1,083,247

In-Sample Elastic Net Regression Results
Optimal l1_ratio: 0.100000
Optimal alpha (regularization): 0.001766
R-squared: 0.000000
RMSE: 0.038577
MSE: 0.001488

Coefficients (on standardized features):
  net_sentiment: 0.000000
  log_volume: 0.000000
  Intercept: -0.000204

Training time: 3.32 seconds


# OOS predictions

In [7]:
# Out-of-sample predictions with MONTHLY TRAINING but DAILY PREDICTIONS
# The model is trained once per month (at month-end) and used to predict all days in the following month

# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW = 252  # Rolling window size in trading days (252 = one year)

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

# Get unique months in the OOS period
oos_months = model_data.loc[model_data['date'] > train_end, 'year_month'].unique()
oos_months = sorted(oos_months)

print(f"Rolling window: {WINDOW} trading days")
print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")
print(f"Number of OOS months: {len(oos_months)}")

def train_and_predict_month(month_idx, pred_month, model_data, unique_dates, oos_dates,
                            FEATURES, TARGET, WINDOW, L1_RATIOS, CV_FOLDS, MAX_ITER):
    """Train Elastic Net model for a single month and generate predictions for all days in that month."""
    predictions = []
    params_info = None
    
    # Get all prediction dates in this month
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    
    if len(month_dates) == 0:
        return predictions, params_info
    
    # Training cutoff: end of the previous month (first day of pred_month - 1 day)
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)
    
    # Find the last trading day before or on train_cutoff
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        return predictions, params_info
    last_train_date = train_dates[-1]
    
    # ROLLING WINDOW: Train on last WINDOW trading days before the month
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx < WINDOW:
        return predictions, params_info
    
    start_date = unique_dates[last_train_date_idx - WINDOW + 1]
    train_mask = (model_data['date'] >= start_date) & (model_data['date'] <= last_train_date)
    X_train = model_data.loc[train_mask, FEATURES]
    y_train = model_data.loc[train_mask, TARGET]
    
    if len(X_train) == 0:
        return predictions, params_info
    
    # Normalize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Fit Elastic Net with CV (use n_jobs=1 inside since we're parallelizing across months)
    enet_model = ElasticNetCV(l1_ratio=L1_RATIOS, cv=CV_FOLDS, random_state=42, n_jobs=1, max_iter=MAX_ITER)
    enet_model.fit(X_train_scaled, y_train)
    params_info = {'month': pred_month, 'alpha': enet_model.alpha_, 'l1_ratio': enet_model.l1_ratio_}
    
    # Use this model to predict for all days in the month
    for pred_date in month_dates:
        test_mask = model_data['date'] == pred_date
        X_test = model_data.loc[test_mask, FEATURES]
        
        if len(X_test) == 0:
            continue
        
        # Transform test features using the same scaler
        X_test_scaled = scaler.transform(X_test)
        
        test_indices = model_data.index[test_mask]
        y_pred = enet_model.predict(X_test_scaled)
        
        for idx, pred in zip(test_indices, y_pred):
            predictions.append({
                'date': pred_date,
                'index': idx,
                'prediction': pred
            })
    
    return predictions, params_info

# Execute in parallel
print(f"\nRunning parallel processing with {N_JOBS} jobs...")
all_results = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(train_and_predict_month)(
        month_idx, pred_month, model_data, unique_dates, oos_dates,
        FEATURES, TARGET, WINDOW, L1_RATIOS, CV_FOLDS, MAX_ITER
    )
    for month_idx, pred_month in enumerate(oos_months)
)

# Collect results
predictions = []
params = []
for month_predictions, params_info in all_results:
    predictions.extend(month_predictions)
    if params_info is not None:
        params.append(params_info)

print(f"\nCompleted.")
print(f"Total predictions: {len(predictions):,}")

Rolling window: 252 trading days
Training window: 2010-01-04 to 2011-12-31
OOS prediction period: 2012-01-03 to 2024-12-30
Number of OOS dates: 3,269
Number of OOS months: 156

Running parallel processing with 16 jobs...


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done   9 tasks      | elapsed:   45.1s
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:  1.4min
[Parallel(n_jobs=16)]: Done  29 tasks      | elapsed:  1.5min
[Parallel(n_jobs=16)]: Done  40 tasks      | elapsed:  2.2min
[Parallel(n_jobs=16)]: Done  53 tasks      | elapsed:  2.9min
[Parallel(n_jobs=16)]: Done  66 tasks      | elapsed:  3.6min
[Parallel(n_jobs=16)]: Done  81 tasks      | elapsed:  4.3min
[Parallel(n_jobs=16)]: Done  96 tasks      | elapsed:  4.4min
[Parallel(n_jobs=16)]: Done 113 tasks      | elapsed:  5.8min
[Parallel(n_jobs=16)]: Done 141 out of 156 | elapsed:  7.1min remaining:   45.1s
[Parallel(n_jobs=16)]: Done 156 out of 156 | elapsed:  7.7min finished



Completed.
Total predictions: 14,545,760


In [8]:
# Summary of selected hyperparameters (alpha and l1_ratio)
print("Selected Hyperparameters Summary")
print("=" * 50)

if params:
    params_df = pd.DataFrame(params)
    print(f"\nRolling {WINDOW}-day Window:")
    print(f"  Alpha - Mean: {params_df['alpha'].mean():.6f}, Std: {params_df['alpha'].std():.6f}")
    print(f"  Alpha - Min: {params_df['alpha'].min():.6f}, Max: {params_df['alpha'].max():.6f}")
    print(f"  L1 ratio - Mean: {params_df['l1_ratio'].mean():.4f}, Std: {params_df['l1_ratio'].std():.4f}")
    print(f"\n  L1 ratio distribution:")
    print(params_df['l1_ratio'].value_counts().sort_index())

Selected Hyperparameters Summary

Rolling 252-day Window:
  Alpha - Mean: 0.000563, Std: 0.001261
  Alpha - Min: 0.000000, Max: 0.009680
  L1 ratio - Mean: 0.1756, Std: 0.2485

  L1 ratio distribution:
l1_ratio
0.10    142
0.25      1
0.95      1
1.00     12
Name: count, dtype: int64


In [9]:
# Convert predictions to DataFrame
predictions_df = pd.DataFrame(predictions)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'prediction']]

# Sort by date and ticker
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"Non-null predictions: {predictions_df['prediction'].notna().sum():,}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))

Predictions Summary
Total observations: 14,545,760
Non-null predictions: 14,545,760

First 10 predictions:
        date  permno ticker    index  prediction
0 2012-01-03   87432      A  2879304   -0.000204
1 2012-01-03   24643     AA  2377768   -0.000204
2 2012-01-03   12479    AAC  2266267   -0.000204
3 2012-01-03   90020   AACC  2994307   -0.000204
4 2012-01-03   15580   AAME  2343833   -0.000204
5 2012-01-03   10517    AAN  2211052   -0.000204
6 2012-01-03   76868   AAON  2576068   -0.000204
7 2012-01-03   89217    AAP  2947508   -0.000204
8 2012-01-03   14593   AAPL  2339391   -0.000204
9 2012-01-03   90854   AATI  3056541   -0.000204


In [10]:
# Save predictions to Data directory
OUTPUT_DATA_DIR = os.path.join(PROJECT_DIR, "Data")
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(OUTPUT_DATA_DIR, f"predictions_elasticnet_input={len(FEATURES)}.pkl")
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")
print(f"Columns: {list(predictions_df.columns)}")

Predictions saved to: C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\predictions_elasticnet_input=2.pkl
File size: 535.60 MB
Shape: (14545760, 5)
Columns: ['date', 'permno', 'ticker', 'index', 'prediction']
